# MCP Client實作

# 選擇模型
請自由任意選擇下面兩個模型之一來進行範例  

## Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

API_KEY = "這邊請改成你自己的API_KEY值"
model_name = 'gemini-2.5-flash'

llm = ChatGoogleGenerativeAI(
    model=model_name,
    google_api_key=API_KEY
)

## LM Studio 或 Ollama

In [ ]:
from langchain_openai import ChatOpenAI
model_name = 'gemma-3-12b-it'  # 指定模型名稱，模型名稱會根據下載的模型不同而改變

# 根據使用LM Studio或Ollama來選擇適當的伺服器URL
base_url = 'http://localhost:1234/v1'  # LM Studio 本地伺服器的URL
# base_url = 'http://localhost:11434/v1' # Ollama 本地伺服器的URL

llm = ChatOpenAI(
    model=model_name,
    openai_api_key="not-needed",
    openai_api_base=base_url 
)

## 測試模型

In [2]:
from langchain_core.messages import HumanMessage
messages = [
    HumanMessage("簡短的說明機器學習的定義")
]
result = llm.invoke(messages)
print(result.content)

機器學習是一種人工智慧技術，讓電腦系統能夠從資料中「學習」，自動識別模式並做出預測或決策，而無需被明確地程式設計。


# 測試用的MCP Server
您可以自行修改 %%writefile 指令後的檔案名稱，以變更程式的儲存位置。

In [2]:
%%writefile c:/data/mcp/mcp_weather.py

from mcp.server.fastmcp import FastMCP

# 建立 MCP Server，名稱可自訂
mcp = FastMCP("weather")

# 匯入 geopy 套件中的 Nominatim 類別，用於地理編碼（將地名轉換成經緯度）
from geopy.geocoders import Nominatim, ArcGIS

# 定義函式：輸入城市名稱，回傳其經緯度座標
@mcp.tool()
def get_coordinates(city_name: str) -> tuple[float, float] | None:
    """
    根據城市名稱取得該城市的緯度和經度。

    Args:
        city_name (str): 欲查詢的城市名稱。

    Returns:
        tuple[float, float] | None: 如果找到城市，則返回包含緯度和經度的元組；
                                   否則返回 None。
    """
    # ---- 第一來源：Nominatim ----
    try:
        geolocator1 = Nominatim(user_agent="clement_fallback_test")
        location = geolocator1.geocode(city_name, timeout=10)

        if location:
            return (location.latitude, location.longitude)
    except:
        pass  # 忽略錯誤，直接進入第二來源

    # ---- 第二來源：ArcGIS  ----
    try:
        geolocator2 = ArcGIS(timeout=10)
        location = geolocator2.geocode(city_name)

        if location:
            return (location.latitude, location.longitude)
    except Exception as e:
        return e

    return None

# 匯入 requests 套件，用於發送 HTTP 請求
import requests

# 定義函式：輸入經緯度，回傳目前天氣資訊（溫度）
@mcp.tool()
def get_weather(latitude: float, longitude: float) -> float:
    """
    根據緯度和經度獲取當前溫度。

    Args:
        latitude (float): 欲查詢地點的緯度。
        longitude (float): 欲查詢地點的經度。

    Returns:
        float: 該地點的當前溫度（攝氏）。
    """

    # 使用 Open-Meteo API 發送 GET 請求，取得氣象資料
    # API 參數：
    # - latitude / longitude: 經緯度
    # - current: 取得目前時刻的溫度 (temperature_2m) 與風速 (wind_speed_10m)
    # - hourly: 取得每小時溫度、相對濕度、風速
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={latitude}&longitude={longitude}&"
        f"current=temperature_2m,wind_speed_10m&"
        f"hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    
    # 將回傳的 JSON 資料解析成 Python 字典
    data = response.json()
    
    # 回傳目前時刻的氣溫（單位：攝氏度）
    return data['current']['temperature_2m']

# 啟動 MCP 伺服器（使用 stdio 模式）
if __name__ == "__main__":
    mcp.run(transport='stdio')

Overwriting c:/data/mcp/mcp_weather.py


MCP Server Config 設定參考
```
{
  "mcpServers": {
    "my-weather-v1":{
      "command": "python",
      "args": [
        "c:/data/mcp/mcp_weather.py"
      ]
    }
  }
}
```

# 顯示MCP Server 訊息

由於程式使用了 asyncio 進行非同步執行，因此無法直接在 Notebook 內執行。  
您可以自行修改 %%writefile 指令後的檔案名稱，以變更程式的儲存位置。

In [21]:
%%writefile c:/data/mcp/mcp_client_display.py
import asyncio
import os

# 從 mcp 套件匯入建立客戶端會話 (ClientSession) 與伺服器連線參數 (StdioServerParameters)
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.shared.metadata_utils import get_display_name

# 定義與伺服器端 (mcp_weather.py) 的連線參數
# 這裡使用標準輸入輸出 (stdio) 模式與 Python 腳本建立通訊
server_params = StdioServerParameters(
    command="python",                         # 使用 python 執行
    args=["c:/data/mcp/mcp_weather.py"],      # 要執行的伺服器端程式
    env=None,                                 # 使用預設環境變數
)


# 非同步函式：列出伺服器端可用的工具 (tools)
async def display_tools(session: ClientSession):
    # 透過 session 取得工具清單
    tools_response = await session.list_tools()

    # 逐一列出每個工具的顯示名稱與描述
    for tool in tools_response.tools:
        display_name = get_display_name(tool)
        print(f"==========================\nTool: {display_name}")
        print('##\nproperties:\n')
        print(f"{tool.inputSchema['properties']}")        
        if tool.description:
            print("##\ndescription:\n")
            print(f"{tool.description}")


# 非同步主要流程：建立 stdio 連線並初始化 MCP session
async def run():
    # 使用 stdio_client 與指定的 server_params 建立雙向通訊通道
    async with stdio_client(server_params) as (read, write):
        # 建立 MCP 客戶端會話
        async with ClientSession(read, write) as session:
            # 初始化會話 (向伺服器發送初始化請求)
            await session.initialize()

            print("=== Available Tools ===")
            # 呼叫上方函式列出所有工具
            await display_tools(session)


# 同步進入點：以 asyncio 執行非同步主流程
def main():
    asyncio.run(run())


# 若此檔案為主程式執行，則呼叫 main()
if __name__ == "__main__":
    main()


Overwriting c:/data/mcp/mcp_client_display.py


In [22]:
# 由於程式將透過作業系統指令執行，請確保執行時使用的檔名與 %%writefile 儲存時的檔名相同。
! python c:/data/mcp/mcp_client_display.py

=== Available Tools ===
Tool: get_coordinates
##
properties:

{'city_name': {'title': 'City Name', 'type': 'string'}}
##
description:


根據城市名稱取得該城市的緯度和經度。

Args:
    city_name (str): 欲查詢的城市名稱。

Returns:
    tuple[float, float] | None: 如果找到城市，則返回包含緯度和經度的元組；
                               否則返回 None。

Tool: get_weather
##
properties:

{'latitude': {'title': 'Latitude', 'type': 'number'}, 'longitude': {'title': 'Longitude', 'type': 'number'}}
##
description:


根據緯度和經度獲取當前溫度。

Args:
    latitude (float): 欲查詢地點的緯度。
    longitude (float): 欲查詢地點的經度。

Returns:
    float: 該地點的當前溫度（攝氏）。



[11/10/25 23:23:06] INFO     Processing request of type           server.py:674
                             ListToolsRequest                                  


# 呼叫MCP Server Tools

## 呼叫 get_coordinates()

In [12]:
%%writefile c:/data/mcp/mcp_client_test.py
import asyncio
import os

# 從 mcp 套件匯入所需模組：
# - ClientSession：建立與 MCP 伺服器的會話
# - StdioServerParameters：定義以標準輸入輸出 (stdio) 方式連線的參數
# - types：定義 MCP 的型別 (例如工具結果、錯誤等)
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client
from mcp.shared.metadata_utils import get_display_name

# 定義與伺服器端 (mcp_weather.py) 的通訊參數
# 此設定會使用 Python 執行該伺服器腳本，並透過 stdin/stdout 通訊
server_params = StdioServerParameters(
    command="python",                        # 指定使用 Python 執行命令
    args=["c:/data/mcp/mcp_weather.py"],     # 指定要啟動的 MCP 伺服器端腳本
    env=None,                                # 使用預設環境變數
)


# 主體的非同步執行邏輯
async def run():
    # 使用 stdio_client() 建立與伺服器的標準輸入輸出通訊管道
    async with stdio_client(server_params) as (read, write):
        # 建立一個 MCP 的客戶端會話 (ClientSession)
        async with ClientSession(read, write) as session:
            # 初始化會話，通知伺服器客戶端已準備好通訊
            await session.initialize()

            # 呼叫伺服器端已註冊的工具 (例如「get_coordinates」)
            # 並傳入參數 city_name='台北'
            result = await session.call_tool(
                "get_coordinates",              # 工具名稱 (由伺服器定義)
                arguments={"city_name": '台北'}  # 傳入的參數 (字典格式)
            )

            # 取得結構化的工具回傳結果 (通常為 JSON 或 dict)
            result_structured = result.structuredContent

            # 印出結果，供使用者檢視
            print(f"結果: {result_structured}")            


# 同步主入口：以 asyncio 執行非同步主程式
def main():
    asyncio.run(run())


# 若此檔案為主執行腳本，則執行 main()
if __name__ == "__main__":
    main()


Overwriting c:/data/mcp/mcp_client_test.py


In [13]:
# 由於程式將透過作業系統指令執行，請確保執行時使用的檔名與 %%writefile 儲存時的檔名相同。
! python c:/data/mcp/mcp_client_test.py

結果: {'result': [25.0375198, 121.5636796]}


[11/10/25 20:02:21] INFO     Processing request of type           server.py:674
                             CallToolRequest                                   
[11/10/25 20:02:22] INFO     Processing request of type           server.py:674
                             ListToolsRequest                                  


## 呼叫 get_weather()

In [15]:
%%writefile c:/data/mcp/mcp_client_test.py
import asyncio
import os

# 從 mcp 模組匯入必要元件：
# - ClientSession：建立與伺服器的客戶端會話
# - StdioServerParameters：設定以標準輸入輸出 (stdio) 為基礎的伺服器連線參數
# - types：MCP 定義的內容型別（例如文字、結構化資料等）
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client
from mcp.shared.metadata_utils import get_display_name

# === MCP 伺服器連線設定 ===
# 這裡定義使用「Python 執行 mcp_weather.py」的伺服器，
# 並透過標準輸入輸出 (stdio) 與之進行雙向通訊。
server_params = StdioServerParameters(
    command="python",                         # 要執行的程式命令
    args=["c:/data/mcp/mcp_weather.py"],      # 要啟動的伺服器端腳本路徑
    env=None,                                 # 使用預設系統環境變數
)


# === 非同步主程式 ===
async def run():
    # 使用 stdio_client() 建立與伺服器的標準輸入輸出通道
    async with stdio_client(server_params) as (read, write):
        # 以建立好的通道初始化一個 MCP 客戶端會話
        async with ClientSession(read, write) as session:
            # 通知伺服器「客戶端已準備好進行通訊」
            await session.initialize()

            # === 呼叫伺服器工具 ===
            # 呼叫伺服器上註冊的工具「get_weather」
            # 並傳入經緯度參數（此例為台北市座標）
            result = await session.call_tool(
                "get_weather", 
                arguments={"latitude": 25.0375198, "longitude": 121.5636796}
            )

            # === 處理伺服器回傳結果 ===
            # 若工具回傳的是結構化資料 (例如 JSON 或字典)，
            # 可透過 structuredContent 直接取得
            result_structured = result.structuredContent
            print(f"結果: {result_structured}")            

# === 同步執行入口 ===
def main():
    asyncio.run(run())  # 使用 asyncio 執行非同步主函式


# === 程式進入點 ===
if __name__ == "__main__":
    main()


Overwriting c:/data/mcp/mcp_client_test.py


In [16]:
# 由於程式將透過作業系統指令執行，請確保執行時使用的檔名與 %%writefile 儲存時的檔名相同。
! python c:/data/mcp/mcp_client_test.py

結果: {'result': 23.0}


[11/10/25 20:04:58] INFO     Processing request of type           server.py:674
                             CallToolRequest                                   
[11/10/25 20:04:59] INFO     Processing request of type           server.py:674
                             ListToolsRequest                                  


# LangChain的MCP工具

In [31]:
%%writefile c:/data/mcp/mcp_langchain_test.py
import asyncio
import os

# 從 mcp 模組匯入必要元件：
# - ClientSession：用來建立與 MCP Server 的客戶端會話
# - StdioServerParameters：用於設定以標準輸入／輸出 (stdio) 為基礎的伺服器啟動參數
# - types：MCP 所定義的各種內容型別（例如文字、結構化資料等）
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

# === MCP 伺服器連線設定 ===
# 這裡指定要啟動的 MCP Server 腳本，並使用「stdio」通訊方式進行互動。
# StdioServerParameters 的作用類似於 JSON 設定檔，用來描述要執行的程式與參數。
server_params = StdioServerParameters(
    command="python",                         # 要執行的主程式命令
    args=["c:/data/mcp/mcp_weather.py"],      # MCP Server 腳本路徑
    env=None,                                 # 沿用系統預設環境變數
)

# === 非同步主程式 ===
async def run():
    # 使用 stdio_client() 與 MCP Server 建立標準輸入／輸出通訊管道
    async with stdio_client(server_params) as (read, write):
        # 建立 MCP 客戶端會話，並與伺服器進行互動
        async with ClientSession(read, write) as session:
            # 初始化會話，通知伺服器「客戶端已準備好」
            await session.initialize()

            # 從 MCP Server 取得可用的工具列表，並轉換成 LangChain 可用格式
            tools = await load_mcp_tools(session)
            print(tools)  # 顯示伺服器提供的工具資訊


# === 同步執行入口 ===
def main():
    # asyncio.run() 用來啟動非同步主程式
    asyncio.run(run())


# === 程式進入點 ===
if __name__ == "__main__":
    main()


Overwriting c:/data/mcp/mcp_langchain_test.py


In [32]:
# 由於程式將透過作業系統指令執行，請確保執行時使用的檔名與 %%writefile 儲存時的檔名相同。
! python c:/data/mcp/mcp_langchain_test.py

[StructuredTool(name='get_coordinates', description='\n根據城市名稱取得該城市的緯度和經度。\n\nArgs:\n    city_name (str): 欲查詢的城市名稱。\n\nReturns:\n    tuple[float, float] | None: 如果找到城市，則返回包含緯度和經度的元組；\n                               否則返回 None。\n', args_schema={'properties': {'city_name': {'title': 'City Name', 'type': 'string'}}, 'required': ['city_name'], 'title': 'get_coordinatesArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x00000168D8715F80>), StructuredTool(name='get_weather', description='\n根據緯度和經度獲取當前溫度。\n\nArgs:\n    latitude (float): 欲查詢地點的緯度。\n    longitude (float): 欲查詢地點的經度。\n\nReturns:\n    float: 該地點的當前溫度（攝氏）。\n', args_schema={'properties': {'latitude': {'title': 'Latitude', 'type': 'number'}, 'longitude': {'title': 'Longitude', 'type': 'number'}}, 'required': ['latitude', 'longitude'], 'title': 'get_weatherArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<fun

[11/11/25 10:50:38] INFO     Processing request of type           server.py:674
                             ListToolsRequest                                  


In [41]:
%%writefile c:/data/mcp/mcp_langchain_test.py
import asyncio
import os

# 從 mcp 模組匯入必要元件：
# - ClientSession：用來建立與 MCP Server 的客戶端會話
# - StdioServerParameters：用於設定以標準輸入／輸出 (stdio) 為基礎的伺服器啟動參數
# - types：MCP 所定義的各種內容型別（例如文字、結構化資料等）
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

# === MCP 伺服器連線設定 ===
# 這裡指定要啟動的 MCP Server 腳本，並使用「stdio」通訊方式進行互動。
# StdioServerParameters 的作用類似於 JSON 設定檔，用來描述要執行的程式與參數。
server_params = StdioServerParameters(
    command="python",                         # 要執行的主程式命令
    args=["c:/data/mcp/mcp_weather.py"],      # MCP Server 腳本路徑
    env=None,                                 # 沿用系統預設環境變數
)

# === 非同步主程式 ===
async def run():
    # 使用 stdio_client() 與 MCP Server 建立標準輸入／輸出通訊管道
    async with stdio_client(server_params) as (read, write):
        # 建立 MCP 客戶端會話，並與伺服器進行互動
        async with ClientSession(read, write) as session:
            # 初始化會話，通知伺服器「客戶端已準備好」
            await session.initialize()

            # 從 MCP Server 取得可用的工具列表，並轉換成 LangChain 可用格式
            tools = await load_mcp_tools(session)
            # 以非同步方式呼叫第一個工具 (get_coordinates)，傳入查詢參數「city_name」
            result = await tools[0].ainvoke({'city_name':'台北'})
            # 輸出執行結果（例如台北市的緯度與經度）
            print(result)


# === 同步執行入口 ===
def main():
    # asyncio.run() 用來啟動非同步主程式
    asyncio.run(run())


# === 程式進入點 ===
if __name__ == "__main__":
    main()


Overwriting c:/data/mcp/mcp_langchain_test.py


In [42]:
# 由於程式將透過作業系統指令執行，請確保執行時使用的檔名與 %%writefile 儲存時的檔名相同。
! python c:/data/mcp/mcp_langchain_test.py

['25.0375198', '121.5636796']


[11/11/25 11:10:31] INFO     Processing request of type           server.py:674
                             ListToolsRequest                                  
                    INFO     Processing request of type           server.py:674
                             CallToolRequest                                   
